# L'éveil du robot — génération vidéo acte par acte

Séquence chorégraphiée en 7 actes (réveil → salut → exploration → étirement → démonstration → célébration → retour au calme), chaque acte étant une commande réelle du Go2 (`webrtc-control`).

Pipeline :
1. Un prompt par acte (limite Omni Flash ≈ 3-10 s par clip)
2. Génération de chaque clip avec `gemini-omni-flash-preview` (clé dans `.env`)
3. **Continuité visuelle** : la dernière frame de chaque clip sert d'image de référence au clip suivant
4. Montage final avec ffmpeg

In [26]:
import base64
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai

ROOT = Path.cwd()  # lancer le notebook depuis la racine du repo Roboss
load_dotenv(ROOT / ".env")

MODEL = "gemini-omni-flash-preview"
OUTDIR = ROOT / "video_parts"
OUTDIR.mkdir(exist_ok=True)

client = genai.Client()  # lit GEMINI_API_KEY depuis l'environnement
print("client prêt, sortie ->", OUTDIR)

client prêt, sortie -> /Users/arresejo/Documents/Projets/evostack/Roboss/video_parts


## 1. Les prompts — un par acte

Style de base commun (identité du robot + décor verrouillés pour la cohérence entre clips), puis l'action de chaque acte. Chaque prompt décrit la **posture de départ** (= posture finale de l'acte précédent) pour des raccords propres.

In [34]:
BASE = ("Realistic 4K footage, eye-level static shot, a Unitree Go2 quadruped robot dog "
        "(appearance strictly as shown in the reference photos), on a smooth concrete "
        "floor in a bright modern lab, soft daylight, shallow depth of field, "
        "no humans in frame, "
        "the robot stays fully visible in frame, no cuts. "
        # --- ancrage anatomique : évite le mélange tête/arrière ---
        "Anatomy anchor, identical in every shot: the HEAD is the unit at the FRONT "
        "bearing a black vertical face stripe with a camera lens and a round "
        "spotlight, a sensor cluster under the chin, and the number 02 on its "
        "sides; the REAR is plain and featureless with no camera, no lights, no "
        "markings. The robot always moves head-first when walking forward. Never "
        "swap or mirror the head and the rear, never show a camera on both ends, "
        "the body never reverses orientation between consecutive moments. "
        # --- contrainte mécanique : la tête du Go2 est FIXE ---
        "Mechanical constraint: the robot's head is rigidly fixed to the body — it "
        "never tilts, pans, nods or moves independently of the torso; only the four "
        "legs articulate, and the whole body moves as one rigid unit. ")

ACTS = [
    ("act1_reveil",
     "The robot dog starts lying flat on its belly with legs folded, head with dark "
     "sensor visor pointing toward the camera, then smoothly extends all four legs and "
     "rises into a tall standing posture, settling into active balance with tiny "
     "micro-adjustments of its legs. 6 seconds."),
    ("act2_salut",
     "The robot dog starts standing, head with dark sensor visor facing the camera. "
     "It sits back slightly onto its hind legs and raises one front leg (next to its "
     "head), waving it in a friendly greeting gesture, then returns to standing. "
     "5 seconds."),
    ("act3_exploration",
     "The robot dog starts standing, head facing the camera. It walks forward "
     "head-first with a natural dog-like gait, then pivots 90 degrees clockwise in "
     "place SLOWLY and smoothly, taking about 3 full seconds — one single continuous "
     "rotation with no cuts, no jumps, every intermediate angle visible — until the "
     "head with the sensor visor points to the right of the frame, then slows into "
     "a careful one-leg-at-a-time walk, still head-first. 8 seconds."),
    ("act4_etirement",
     "The robot dog starts standing in side profile, head with sensor visor on the "
     "right of the frame. It bows deeply into a play-bow stretch: BOTH front legs "
     "fully extended forward and flat on the ground, chest lowered between them, "
     "while the featureless back half stays raised on the hind legs, then it rises "
     "back to standing. 5 seconds."),
    ("act7_calme",
     "The robot dog starts standing in side profile, head with sensor visor on the "
     "right of the frame. It lowers its back half into a dog-like sitting pose "
     "(head up), pauses for a moment, then pushes back up with its hind legs and "
     "returns to a tall, stable standing posture, ending motionless and upright. "
     "6 seconds."),
]

for name, action in ACTS:
    print(f"{name:>20} : {action[:70]}...")


         act1_reveil : The robot dog starts lying flat on its belly with legs folded, head wi...
          act2_salut : The robot dog starts standing, head with dark sensor visor facing the ...
    act3_exploration : The robot dog starts standing, head facing the camera. It walks forwar...
      act4_etirement : The robot dog starts standing in side profile, head with sensor visor ...
          act7_calme : The robot dog starts standing in side profile, head with sensor visor ...


## 2. Fonctions de génération

- `generate_clip` : appel Omni Flash (livraison URI pour les clips &gt; 4 Mo), avec retry
- `last_frame` : extrait la dernière frame d'un clip (ffmpeg) pour ancrer le clip suivant

In [28]:
import mimetypes

# --- photos du vrai Go2 Pro (identité) : déposer des .jpg/.png dans robot_refs/
REFS_DIR = ROOT / "robot_refs"
REFS_DIR.mkdir(exist_ok=True)
ID_PHOTOS = sorted(p for p in REFS_DIR.iterdir()
                   if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))[:3]
print(f"{len(ID_PHOTOS)} photo(s) d'identité :", [p.name for p in ID_PHOTOS])


def _img_part(path: Path) -> dict:
    return {"type": "image",
            "data": base64.b64encode(path.read_bytes()).decode(),
            "mime_type": mimetypes.guess_type(path)[0] or "image/png"}


def _download_via_uri(uri: str, output: Path) -> None:
    file_name = uri.split("/")[-1].split(":")[0]
    while True:
        info = client.files.get(name=f"files/{file_name}")
        if info.state.name == "ACTIVE":
            break
        if info.state.name == "FAILED":
            raise RuntimeError("Video processing failed")
        time.sleep(5)
    output.write_bytes(client.files.download(file=uri))


def generate_clip(prompt: str, output: Path, ref_image: Path | None = None,
                  retries: int = 1) -> Path:
    parts = []
    prefix = ""
    # 1) photos d'identité du vrai robot (toujours en premier)
    if ID_PHOTOS:
        parts += [_img_part(p) for p in ID_PHOTOS]
        prefix += (f"The first {len(ID_PHOTOS)} image(s) are photos of the EXACT "
                   "robot that must appear in the video: reproduce its precise "
                   "appearance — colors, materials, proportions, head unit, "
                   "leg design, markings — do not idealize or restyle it. ")
    # 2) dernière frame du clip précédent (continuité)
    if ref_image is not None:
        parts.append(_img_part(ref_image))
        prefix += ("The last image is the final frame of the previous shot: "
                   "continue exactly from it (same robot pose, same room, same "
                   "lighting, same camera position). In that frame, identify "
                   "the robot's HEAD as the end bearing the dark sensor visor — "
                   "keep that same end as the head for the entire clip, never "
                   "mirror or flip the robot's orientation. ")
    parts.append({"type": "text", "text": prefix + prompt})

    last_err = None
    for attempt in range(retries + 1):
        try:
            interaction = client.interactions.create(
                model=MODEL,
                input=parts if len(parts) > 1 else prompt,
                response_format={"type": "video", "aspect_ratio": "16:9",
                                 "delivery": "uri"},
            )
            video = interaction.output_video
            _download_via_uri(video.uri, output)
            return output
        except Exception as e:
            last_err = e
            print(f"  tentative {attempt + 1} échouée : {e}")
            time.sleep(10)
    raise last_err


def last_frame(video: Path, output: Path) -> Path:
    subprocess.run(
        ["ffmpeg", "-y", "-v", "error", "-sseof", "-0.2", "-i", str(video),
         "-frames:v", "1", "-q:v", "2", str(output)],
        check=True,
    )
    return output


2 photo(s) d'identité : ['unitree-go2-pro-ai-quadruped-robot-dog-452067.webp', 'unitree-go2-x-robotic-dog-04_94674ce5-1bd3-45f3-9eb6-6570e5d576a0.webp']


## 3. Génération des 7 clips

Chaque clip démarre visuellement là où le précédent s'est arrêté (dernière frame en référence). Compter ~1-3 min par clip.

In [35]:
clips = []
ref = None  # pas de référence pour l'acte 1

for name, action in ACTS:
    out = OUTDIR / f"{name}.mp4"
    if out.exists():
        print(f"{name}: déjà généré, on passe ({out.stat().st_size//1024} KB)")
    else:
        print(f"{name}: génération...")
        t0 = time.time()
        generate_clip(BASE + action, out, ref_image=ref)
        print(f"{name}: OK en {time.time()-t0:.0f}s ({out.stat().st_size//1024} KB)")
    clips.append(out)
    ref = last_frame(out, OUTDIR / f"{name}_last.png")

print(f"\n{len(clips)} clips prêts")

act1_reveil: déjà généré, on passe (1478 KB)
act2_salut: déjà généré, on passe (1218 KB)
act3_exploration: déjà généré, on passe (2013 KB)
act4_etirement: déjà généré, on passe (1278 KB)
act7_calme: génération...
act7_calme: OK en 43s (1505 KB)

5 clips prêts


## 4. Montage final

Concaténation ffmpeg avec ré-encodage (raccords propres même si les paramètres d'encodage diffèrent entre clips).

In [36]:
FINAL = ROOT / "eveil_du_robot.mp4"

concat_list = OUTDIR / "concat.txt"
concat_list.write_text("".join(f"file '{c.resolve()}'\n" for c in clips))

subprocess.run(
    ["ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0",
     "-i", str(concat_list),
     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", "24",
     "-an", str(FINAL)],
    check=True,
)
dur = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "csv=p=0", str(FINAL)],
    capture_output=True, text=True).stdout.strip()
print(f"Vidéo finale : {FINAL} ({float(dur):.1f}s, {FINAL.stat().st_size//1024} KB)")

Vidéo finale : /Users/arresejo/Documents/Projets/evostack/Roboss/eveil_du_robot.mp4 (30.1s, 3126 KB)


In [20]:
# aperçu dans le notebook
from IPython.display import Video
Video(str(FINAL), width=720)

## 5. Plan exact — depuis la chorégraphie, pas depuis les pixels

Le tracker vidéo ne récupère que locomotion/postures (timing ~0.25 s, tricks non détectés). Or la chorégraphie est **connue** : chaque acte correspond à des commandes précises. On construit donc le plan directement depuis la partition, avec les débuts d'actes mesurés sur les durées réelles des clips (ffprobe) — précision parfaite, tricks inclus.


In [37]:
import json

def clip_duration(path: Path) -> float:
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", str(path)],
        capture_output=True, text=True, check=True).stdout.strip()
    return float(out)

# débuts d'actes = offsets cumulés des durées réelles des clips
durations = {name: clip_duration(OUTDIR / f"{name}.mp4") for name, _ in ACTS}
starts, t = {}, 0.0
for name, _ in ACTS:
    starts[name] = t
    t += durations[name]
print({k: f"{v:.1f}s" for k, v in starts.items()})

# la partition : commandes exactes de chaque acte (offsets relatifs à l'acte)
CHOREO = {
    "act1_reveil":        [(0.0, {"action": "sport", "cmd": "StandDown"}),
                           (2.0, {"action": "sport", "cmd": "StandUp"}),
                           (3.5, {"action": "sport", "cmd": "BalanceStand"})],
    "act2_salut":         [(0.5, {"action": "sport", "cmd": "Hello"})],
    "act3_exploration":   [(0.0, {"action": "move", "x": 0.4, "y": 0.0, "z": 0.0, "duration": 3.0}),
                           # rotation lente — z réduit pour coller au pivot ~3s du clip
                           (3.5, {"action": "move", "x": 0.0, "y": 0.0, "z": -0.5, "duration": 3.2}),
                           (7.0, {"action": "move", "x": 0.25, "y": 0.0, "z": 0.0, "duration": 1.0})],
    "act4_etirement":     [(0.5, {"action": "sport", "cmd": "Stretch"})],
    "act7_calme":         [(0.5, {"action": "sport", "cmd": "Sit"}),
                           # retour debout — l'animation Sit dure ~3s
                           (5.0, {"action": "sport", "cmd": "RiseSit"}),
                           # stabilisation finale en station active
                           (9.0, {"action": "sport", "cmd": "BalanceStand"})],
}

steps = []
for name, entries in CHOREO.items():
    for off, cmd in entries:
        steps.append({"t": round(starts[name] + off, 2), **cmd})
steps.sort(key=lambda s: s["t"])

PLAN = ROOT / "robotdog" / "webrtc-control" / "plan_eveil_exact.json"
PLAN.write_text(json.dumps({"steps": steps, "source": "choreography"}, indent=2))
print(f"{len(steps)} steps -> {PLAN}")
for s in steps:
    desc = s.get("cmd") or "move x={:+.2f} z={:+.2f} ({}s)".format(s["x"], s["z"], s["duration"])
    print(f"  t={s['t']:6.2f}s  {desc}")


{'act1_reveil': '0.0s', 'act2_salut': '6.0s', 'act3_exploration': '11.0s', 'act4_etirement': '19.0s', 'act7_calme': '24.0s'}
11 steps -> /Users/arresejo/Documents/Projets/evostack/Roboss/robotdog/webrtc-control/plan_eveil_exact.json
  t=  0.00s  StandDown
  t=  2.00s  StandUp
  t=  3.50s  BalanceStand
  t=  6.52s  Hello
  t= 11.03s  move x=+0.40 z=+0.00 (3.0s)
  t= 14.53s  move x=+0.00 z=-0.50 (3.2s)
  t= 18.03s  move x=+0.25 z=+0.00 (1.0s)
  t= 19.53s  Stretch
  t= 24.54s  Sit
  t= 29.04s  RiseSit
  t= 33.04s  BalanceStand


## 5. (Bonus) Rejouer la séquence sur le vrai robot

La vidéo finale peut repasser par le pipeline vidéo→robot :

```bash
cd robotdog/webrtc-control
/Users/arresejo/Documents/Projets/evostack/Roboss/.venv/bin/python video_to_motion.py analyze ../../eveil_du_robot.mp4 -o plan_eveil.json
./.venv/bin/python video_to_motion.py replay plan_eveil.json --dry-run
./.venv/bin/python video_to_motion.py replay plan_eveil.json --no-avoid
```

Note : les tricks (Hello, Stretch, MoonWalk, Dance) ne sont pas détectés par le tracker (postures/marche/rotations uniquement) — pour l'exécution fidèle, utiliser directement la séquence de commandes connue : `StandUp → BalanceStand → Hello → ClassicWalk → yaw → StaticWalk → Stretch → TrotRun → Stop → MoonWalk → WiggleHips → Dance1 → Sit → Damp`, avec ~2 s de pause entre chaque (le Go2 avale les commandes envoyées pendant une animation).